# penRed visualization and particle tracking v2

In [ ]:
# import libs
import pyPenred
from pathlib import Path
import os

In [ ]:
config = pyPenred.readConfigFile("config.in") # Load the file as a dictionary
config['simulation']['threads'] = 32  # Modify parameters
config['sources']['generic']['source1']['nhist'] = 1.0e7
config['geometry']['input-file'] = "disc_bse.geo"
pyPenred.writeConfigFile("modified.in", config)

In [ ]:
#run the sim
pyPenred.runFromFile("modified.in")

## Notes

Need to create actual detector regions around the solid, need to create a material for the detector regions (should be a vacuum), PSF for particle tracking and what not

## animation

In [ ]:
import pyPenred
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from pathlib import Path

folder = Path(r"E:\Code\EPBF_Twin\testing\particlev2")

psf_file = folder / "psf-merged.dat"
ascii_file = folder / "psf-electrons.txt"

# Convert binary PSF to ASCII
pyPenred.psf.psf2ascii(
    str(psf_file),
    str(ascii_file)
)

columns = [
    "DHIST", "KPAR", "E", "X", "Y", "Z",
    "U", "V", "W", "WGHT", "PAGE",
    "ILB1", "ILB2", "ILB3", "ILB4", "ILB5"
]

df = pd.read_csv(
    ascii_file,
    sep=r"\s+",
    comment="#",
    header=None,
    names=columns
)

# Keep electrons entering the scoring slab moving away from the surface (W < 0).
# Drops primaries crossing the slab on the way in if the source sits above it.
df = df[(df["KPAR"] == 0) & (df["W"] < 0)].copy()

# Convert units
df["E_keV"] = df["E"] / 1000
df["X_um"] = df["X"] * 1e4
df["Y_um"] = df["Y"] * 1e4
df["Z_um"] = df["Z"] * 1e4

print("Recorded electrons:", len(df))
print(df.head())

In [ ]:
from pathlib import Path

folder = Path(r"E:\Code\EPBF_Twin\testing\particlev2")

for file in sorted(folder.glob("*psf*")):
    print(file.name, file.stat().st_size, "bytes")

## geometry + trajectories figure

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from pathlib import Path

folder = Path(r"E:\Code\EPBF_Twin\testing\particlev2")
L = 5.0        # half-width of the view in um (the Cu disc itself is 200 um across)
N_TRACKS = 40  # primary electrons to draw


def box(ax, x, y, z, color, alpha, zorder):
    (x0, x1), (y0, y1), (z0, z1) = x, y, z
    v = np.array([[x0, y0, z0], [x1, y0, z0], [x1, y1, z0], [x0, y1, z0],
                  [x0, y0, z1], [x1, y0, z1], [x1, y1, z1], [x0, y1, z1]])
    faces = [[0, 1, 2, 3], [4, 5, 6, 7], [0, 1, 5, 4], [2, 3, 7, 6], [1, 2, 6, 5], [0, 3, 7, 4]]
    ax.add_collection3d(Poly3DCollection([v[f] for f in faces], facecolor=color, alpha=alpha,
                                         edgecolor="#3a3a38", linewidth=0.6, zorder=zorder))


# Each particle is a block separated by two blank lines.
# Columns: E X Y Z U V W WGHT IBODY MAT PAGE ILB1..5  (IBODY 1 = scoring slab, ILB1 1 = primary)
tracks = []
for f in sorted(folder.glob("track-th*-electron-tracks.dat")):
    for block in f.read_text().split("\n\n\n"):
        rows = [l.split() for l in block.splitlines() if l.strip() and not l.startswith("#")]
        if rows and rows[0][11] == "1":
            t = np.array(rows, float)
            if np.abs(t[:, 1:3]).max() * 1e4 < L:  # keep tracks that fit in the view
                tracks.append(t)
tracks = tracks[:N_TRACKS]

fig = plt.figure(figsize=(7, 6), facecolor="white")
ax = fig.add_subplot(projection="3d", computed_zorder=False)

# Geometry, plotted with height = -z so the beam comes from above (um)
box(ax, (-L, L), (-L, L), (-4, 0), "#c87f4f", 0.25, zorder=1)   # Cu disc (section)
box(ax, (-L, L), (-L, L), (2, 3), "#b8c4cc", 0.30, zorder=3)    # BSE scoring slab

n_bse = 0
for t in tracks:
    bse = ((t[:, 8] == 1) & (t[:, 6] < 0)).any()  # entered the slab moving away from the surface
    n_bse += bse
    ax.plot(t[:, 1] * 1e4, t[:, 2] * 1e4, -t[:, 3] * 1e4, zorder=2,
            color="#2a78d6" if bse else "#77766f", lw=1.0 if bse else 0.6)

ax.text(L, -L, -4, " Cu", color="#0b0b0b")
ax.text(L, -L, 3, " BSE scoring slab", color="#0b0b0b")

ax.set(xlim=(-L, L), ylim=(-L, L), zlim=(-4, 3))
ax.set_box_aspect((2 * L, 2 * L, 7))
ax.view_init(elev=18, azim=-60)
ax.set_axis_off()
ax.legend(handles=[Line2D([], [], color="#2a78d6", lw=1.5, label=f"Backscattered ({n_bse})"),
                   Line2D([], [], color="#77766f", lw=1.5, label=f"Absorbed ({len(tracks) - n_bse})")],
          loc="upper left", frameon=False)
ax.set_title("40 keV electrons on Cu", loc="left", color="#0b0b0b")
fig.text(0.5, 0.17, f"View cropped to ±{L:g} µm; full disc is Ø200 µm × 50 µm", ha="center", color="#52514e", fontsize=9)

fig.savefig(folder / "bse_trajectories.png", dpi=200, bbox_inches="tight")
plt.show()